# A.5: Packing a corpus into training blocks

Join documents with end-of-text tokens, save the stream as 16-bit integers, and draw input and target blocks.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From A.1 tiktoken basics
import tiktoken


In [ ]:
import numpy as np
import torch

enc = tiktoken.get_encoding("gpt2")
docs = ["First document.", "Second, slightly longer document.", "Third."]
stream = []
for d in docs:
    stream.extend(enc.encode_ordinary(d))
    stream.append(enc.eot_token)                 # <|endoftext|>, ID 50256
print(len(stream), stream)

np.array(stream, dtype=np.uint16).tofile("corpus.bin")   # GPT-2 IDs fit in 16 bits
data = torch.from_numpy(np.fromfile("corpus.bin", dtype=np.uint16).astype(np.int64))

def get_batch(data, batch_size, block_size, generator=None):
    starts = torch.randint(len(data) - block_size, (batch_size,), generator=generator)
    x = torch.stack([data[s:s + block_size] for s in starts])
    y = torch.stack([data[s + 1:s + block_size + 1] for s in starts])
    return x, y

g = torch.Generator().manual_seed(0)
x, y = get_batch(data, batch_size=2, block_size=8, generator=g)
print(x)
print(y)
print(repr(enc.decode(x[0].tolist())), "->", repr(enc.decode(y[0].tolist())))


**Expected output**

Output:

```text
14 [5962, 3188, 13, 50256, 12211, 11, 4622, 2392, 3188, 13, 50256, 22747, 13, 50256]
tensor([[   13, 50256, 12211,    11,  4622,  2392,  3188,    13],
        [50256, 12211,    11,  4622,  2392,  3188,    13, 50256]])
tensor([[50256, 12211,    11,  4622,  2392,  3188,    13, 50256],
        [12211,    11,  4622,  2392,  3188,    13, 50256, 22747]])
'.<|endoftext|>Second, slightly longer document.' -> '<|endoftext|>Second, slightly longer document.<|endoftext|>'
```
